# AI Product Intelligence System
### Gen AI Bootcamp - Day 2 Homework Challenge

This notebook implements the three tasks required for the Day 2 Homework Challenge:
1. **Task 1: Smart Product Recommendation Engine** (Suggesting complementary items)
2. **Task 2: Unique Product Catalog Creation** (Grouping duplicates using CLIP embeddings)
3. **Task 3: Reverse Product Search** (Text-to-Image search using CLIP)

## Setup and Preprocessing

We will load the dataset programmatically, filter to items with valid images, and set up our CLIP embedding environment.

In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
import torch
from PIL import Image
import kagglehub
from sentence_transformers import SentenceTransformer
from IPython.display import display, Image as IPImage
import matplotlib.pyplot as plt

# 1. Download/Retrieve Dataset
print("Retrieving dataset path...")
dataset_dir = kagglehub.dataset_download('paramaggarwal/fashion-product-images-small')
print(f"Dataset location: {dataset_dir}")

# 2. Load styles.csv with error-skipping for malformed lines
csv_path = os.path.join(dataset_dir, "styles.csv")
df_raw = pd.read_csv(csv_path, on_bad_lines='skip')
df_raw['id'] = df_raw['id'].astype(str)
images_dir = os.path.join(dataset_dir, "images")
df_raw['image_path'] = df_raw['id'].apply(lambda x: os.path.join(images_dir, f'{x}.jpg'))

# Filter to existing files only
df = df_raw[df_raw['image_path'].apply(os.path.exists)].reset_index(drop=True)
print(f"Total items with images: {len(df)}")

### Feature Extraction (CLIP Embeddings)

We will sample a subset of the dataset (e.g. 3,000 items) to keep computation fast while maintaining a rich set of categories. We will use the pre-trained `clip-ViT-B-32` model.

In [ ]:
from src.utils import get_or_create_embeddings

# Load or generate CLIP embeddings (default sample size: 3000)
df_sampled, image_embeddings, text_embeddings = get_or_create_embeddings(df, sample_size=3000)
print(f"Sample size: {len(df_sampled)}")
print(f"Image embeddings shape: {image_embeddings.shape}")
print(f"Text embeddings shape: {text_embeddings.shape}")

## Task 1: Smart Product Recommendation Engine

Visual search finds items that are *visually identical*. But e-commerce systems need to recommend **complementary** items (e.g., Running Shoes suggest Socks, Fitness Watch, Sports Bottle). 

**Our Approach:** We define rules linking categories to matching complementary categories, filter candidates for gender and usage (e.g. Sports vs. Casual), and then rank candidates using CLIP embedding similarity.

In [ ]:
from src.recommender import get_complementary_recommendations

# Let's find a running shoe in the dataset to test
shoes = df_sampled[df_sampled['subCategory'] == 'Shoes']
sports_shoes = shoes[shoes['usage'] == 'Sports']

if len(sports_shoes) > 0:
    sample_shoe = sports_shoes.iloc[0]
else:
    sample_shoe = shoes.iloc[0]
    
print(f"Input Product: {sample_shoe['productDisplayName']} (ID: {sample_shoe['id']})")
display(IPImage(filename=sample_shoe['image_path'], width=150))

print("\n--- Generating Complementary Recommendations ---")
recs = get_complementary_recommendations(sample_shoe['id'], df_sampled, image_embeddings, top_k=3)

for idx, rec in enumerate(recs):
    print(f"\nRecommendation {idx+1}: {rec['relationship']} ({rec['productDisplayName']})")
    print(f"  Match Score: {rec['score']:.4f} | Gender: {rec['gender']} | Color: {rec['baseColour']}")
    display(IPImage(filename=rec['image_path'], width=120))

## Task 2: Unique Product Catalog Creation

Marketplaces often have redundant or near-duplicate listings upload by multiple sellers. 

**Our Approach:** We group items using leader clustering based on a high cosine similarity threshold (e.g., 0.88). For each group, we compute the medoid representative, clean the display names of trailing tokens/packs, and output a clean, unique product catalog.

In [ ]:
from src.catalog import create_unique_catalog

# Create the unique catalog
unique_df, mapping, duplicates, stats = create_unique_catalog(df_sampled, image_embeddings, similarity_threshold=0.88)

print("--- Catalog Deduplication Summary Statistics ---")
for key, val in stats.items():
    print(f"{key.replace('_', ' ').title()}: {val}")

# Print some of the duplicate groups identified
print("\n--- Sample Duplicate Groups Found ---")
for idx, cluster in enumerate(duplicates[:3]):
    print(f"\nCluster {idx+1} (Representative: {cluster['representative_name']})")
    print(f"Contains {len(cluster['items'])} items:")
    for item in cluster['items']:
        print(f" - ID: {item['id']} | Name: {item['productDisplayName']}")
        display(IPImage(filename=item['image_path'], width=100))

## Task 3: Reverse Product Search

Allow users to search the database using free-text queries. 

**Our Approach:** Encode the text query into the CLIP shared text-image space, and compute cosine similarity against the pre-computed image embeddings.

In [ ]:
from src.search import reverse_product_search

# Define search queries to test
queries = [
    "blue casual shirt",
    "red athletic shoes",
    "leather ladies handbag"
]

for query in queries:
    print(f"\n\nQuery: '{query}'")
    print("="*40)
    results = reverse_product_search(query, df_sampled, image_embeddings, top_k=4)
    
    # Plot results in a grid
    fig, axes = plt.subplots(1, len(results), figsize=(12, 3))
    if len(results) == 1:
        axes = [axes]
        
    for ax, res in zip(axes, results):
        img = Image.open(res['image_path'])
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f"{res['productDisplayName'][:15]}...\nMatch: {res['score']*100:.1f}%", fontsize=9)
    plt.show()